# Multiquery RAG -- Production Version

Replaces notebooks 5 (`MultiQ+Rerank+Generate RAG Pipleline`) and 6
(`Advanced RAG -- CostumeMultiQ+Critiq-Rerank+Generate`) with the
production implementation now living in `multiquery.py` and
`rag_pipeline.py`'s `query_multi()`.

**What changed from the 2023 version:**
- LCEL (`prompt | llm | parser`) instead of the deprecated `LLMChain`
- No `langchain.retrievers.MultiQueryRetriever` -- its generic document
  dedup has no awareness of this project's `max_per_title` cap or
  section metadata, so it would have regressed the source-noise fixes
  already validated in the naive-vs-rerank eval
- Sub-query retrieval results are pooled and deduped by
  `(title, section, chunk)`, then run through the *same* rerank +
  dedup logic `query()` already uses (`_rerank_and_dedup()`), not a
  second, divergent copy of it
- The original question is always included alongside the reformulated
  ones, as a safety net against the LLM's reformulations drifting
- **The number of reformulations is decided by the model itself**, not
  fixed. `max_subqueries` is a cost-control ceiling, not a target -- a
  simple question typically gets 1 reformulation, a genuinely composite
  one gets more, up to the cap. A fixed count would either pad simple
  questions with redundant paraphrases or under-serve genuinely complex
  ones.

**Why this exists:** Part 1 and Part 3 of the blog series found that
composite questions (ones needing two distant parts of an article)
sometimes fail because a single embedding skews toward whichever half
of the question is semantically "louder," starving the other half of
retrieved content. Multiquery is the direct fix -- each half gets its
own retrieval pass with a clean embedding target.

**Test set:** tests 1-4 use the original blog-eval questions (Cicero
vs. Socrates, a simple baseline, the African Sage Philosophy drift
case). Tests 5-7 add three more cross-section composites, each verified
against real indexed article text. Tests 8-9 are deliberately harder,
multi-article demo questions built to contrast against ungrounded
chatbots -- see the note before them; they aren't scored the same way
as the rest.

Every question in this notebook references only articles confirmed
present in the corpus -- checked directly against `data/SEP.parquet`
before writing any of them, after an earlier draft of this notebook
included questions about articles (Aristotle, Stoicism, Seneca) that
turned out not to be indexed at all.


In [1]:
# This notebook lives in notebooks/, one level below the project root.
# Same fix as the chunking-comparison notebook: add the root to sys.path
# and chdir into it so both imports and relative file paths resolve.
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print("Working directory set to:", os.getcwd())


Working directory set to: c:\Users\user\Dropbox\Career\Tech\++Projects\DeepAnalytic\SEP


In [2]:
from rag_pipeline import RerankRAG
from multiquery import generate_subqueries

rag = RerankRAG()
print("RerankRAG initialized.")

c:\Users\user\anaconda3\envs\deepanalytic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RerankRAG initialized.


## Test 1: sanity check -- what do the reformulated queries actually look like?

Before running anything through retrieval, just look at what
`generate_subqueries()` produces for a known composite question, the
one from the blog series that naive/rerank both failed on identically.
Note this question should get more than 1 reformulation, since it's
genuinely composite; a simple question run through the same function
should come back with just 1 or 2.


In [3]:
import inspect
from multiquery import generate_subqueries
print(inspect.signature(generate_subqueries))

(question: str, max_n: int = 5, return_topics: bool = False)


In [4]:
question = "How does Cicero's res publica compare to Socrates' relationship to the Laws of Athens at his trial?"

subqueries = generate_subqueries(question, max_n=5)
print("Reformulated queries:")
for q in subqueries:
    print(" -", q)


Reformulated queries:
 - What are the main principles of Cicero's res publica?
 - What was Socrates' stance on the Laws of Athens during his trial?
 - How does Cicero's res publica compare to Socrates' relationship to the Laws of Athens at his trial?


## Test 2: the known composite-question failure

This is the exact question that failed identically across every
chunking strategy in Part 2/3 -- naive and rerank both retrieved
Socrates/trial content almost exclusively, with the res publica
definition either missing or present-but-unused. Compare `query()`
against `query_multi()` on the identical question.


In [5]:
result_single = rag.query(question)
print("=== query() (single embedding) ===")
print(result_single["answer"])
print()
print("Sources:", [(s.get("title"), s.get("section")) for s in result_single["sources"]])


=== query() (single embedding) ===
The context does not provide any information about Cicero's res publica, so it is not possible to compare it to Socrates' relationship to the Laws of Athens at his trial using only the facts given.

Sources: [('Ancient Political Philosophy', 'Socrates and Plato'), ('Ancient Political Philosophy', 'Socrates and Plato')]


#### query() misses Cicero content entirely on this run

This run of the single-embedding baseline retrieved zero content from the Cicero section. Both returned sources came from "Socrates and Plato," and none came from "The Roman Republic and Cicero," which is the section containing the actual res publica definition. The answer correctly reflects this gap rather than hallucinating around it, stating plainly that the context doesn't provide information about Cicero's res publica, but the underlying problem is upstream of generation this time: retrieval itself didn't surface the content at all, rather than surfacing it and then losing it during reranking, which is what happened in earlier runs of this same question. That's a meaningfully different and more basic failure than the one documented earlier in this log, and it's worth sitting with rather than smoothing over, since it suggests single-query retrieval on this particular question isn't fully stable run to run, at least not stable enough to trust as a one-time test.

In [6]:
result_multi = rag.query_multi(question, max_subqueries=5)
print("=== query_multi() ===")
print("Subqueries used:", result_multi["subqueries"])
print()
print(result_multi["answer"])
print()
print("Sources:", [(s.get("title"), s.get("section")) for s in result_multi["sources"]])


=== query_multi() ===
Subqueries used: ["What are the main principles of Cicero's res publica?", "What was Socrates' stance on the Laws of Athens during his trial?", "How does Cicero's res publica compare to Socrates' relationship to the Laws of Athens at his trial?"]

Cicero's concept of the res publica, or "commonwealth," is defined as the "property/thing of the people" and is based on the idea of a people being an assemblage united by agreement on law and community of interest. This implies a collective ownership and responsibility for the common good, with magistrates entrusted to use it for the people's welfare.

In contrast, Socrates' relationship to the Laws of Athens, as depicted in the context of his trial, is characterized by an unequal social contract. The Laws compare themselves to parents and slaveowners, with Socrates as a child and slave, suggesting a hierarchical relationship where obedience is owed due to the Laws providing the basis for Socrates' life and education in

Let's see what were the consulted chunks.

In [7]:
result = rag.query_multi(question, max_subqueries=5)
for s in result["sources"]:
    print(f"--- {s.get('title')} | {s.get('section')} ---")
    print(s.get("text", "")[:500])
    print()

--- Ancient Political Philosophy | The Roman Republic and Cicero ---
While justice was for Cicero, as for the Greeks, the fundamental bond
of the commonwealth, he offered distinctive and influential linked
definitions of the res publica (”commonwealth“,
or more literally the ”public thing“) as the res
populi, the ”property/thing of the people“, and of a
populus or ”people“ in turn as ”an
assemblage associated with one another by agreement on
“law” [iuris consensu] and community of interest
[utilitatis communione]“ (both, I.39). This could be
interpreted either as

--- Ancient Political Philosophy | Socrates and Plato ---
The “Laws of Athens” appeal to a kind of social contract
made between themselves and Socrates. The contract is unequal: the
“Laws” compare themselves to parents and slaveowners, and
Socrates to child and slave. Obedience to them is owed, they claim,
because the “Laws” have provided the whole basis for
Socrates’ education and life in the city, a city in which he has
not

In [8]:
r_floor = rag.query_multi(question, max_subqueries=5)
r_concat = rag.query_multi_concat(question, max_subqueries=5, k_per_subquery=3)

print("=== query_multi() -- rerank + floor + cap fix ===")
print(r_floor["answer"])
print()
print("=== query_multi_concat() -- simple, no rerank, no cap ===")
print(r_concat["answer"])

=== query_multi() -- rerank + floor + cap fix ===
Cicero's concept of the res publica, or "commonwealth," is defined as the "property/thing of the people" and is based on the idea of a people being an assemblage united by agreement on law and community of interest. This implies a collective ownership and responsibility for the common good, with magistrates entrusted to use it for the people's welfare.

In contrast, Socrates' relationship to the Laws of Athens, as depicted in the context of his trial, is characterized by an unequal social contract. The Laws compare themselves to parents and slaveowners, with Socrates as a child and slave, suggesting a hierarchical relationship where obedience is owed due to the Laws providing the basis for Socrates' life and education in the city. However, the Laws also offer Socrates the opportunity to "persuade or obey" them, indicating a limited form of agency within this relationship.

In summary, Cicero's res publica emphasizes collective ownership

#### Multiquery reliably finds and synthesizes both halves, again
Both `query_multi()` and `query_multi_concat()` correctly retrieved content from both the Cicero section and the Socrates section on this run, the same reliable result seen in earlier testing of this question, and both produced genuine synthesized comparisons rather than the refusal that used to be the default outcome here before the generation prompt was fixed. The reranked version stayed a little thinner, built mostly from the two floor-guaranteed chunks, while the concat version pulled in a third genuinely relevant passage, the Crito and Republic discussion of justice, and used it to add real substance: Cicero's point that a regime lacking justice isn't truly a res publica at all, and Socrates' own principle that it's better to suffer injustice than to commit it. Neither of those specific points made it into the reranked version's answer. The pattern from earlier in this log held again here: when single-query retrieval was unreliable on this exact question, in this exact run, multiquery wasn't just better, it was the only one of the three methods that worked at all.

## Test 3: does multiquery help or hurt a simple, single-topic question?

Multiquery shouldn't be the default for every question -- it costs an
extra LLM call plus N retrieval calls. Worth checking it doesn't
*regress* a question that `query()` already handles well.


In [9]:
simple_question = "What is the thinking animal argument?"

result_simple_single = rag.query(simple_question)
result_simple_multi = rag.query_multi(simple_question, max_subqueries=5)

print("=== query() ===")
print(result_simple_single["answer"])
print()
print("=== query_multi() ===")
print("Subqueries:", result_simple_multi["subqueries"])
print(result_simple_multi["answer"])


=== query() ===
The thinking animal argument is a philosophical argument that suggests the following:

1. (P1) Presently sitting in your chair is a human animal.
2. (P2) The human animal sitting in your chair is thinking.
3. (P3) You are the thinking being sitting in your chair.
4. (C) Therefore, the human animal sitting in your chair is you.

The argument is used to explore the concept of personal identity and the relationship between humans and animals. It has been developed and discussed by various philosophers, including Snowdon, Carter, McDowell, Ayers, and Olson. The argument raises issues about the identity of thinking beings and the potential problem of being identical with multiple nonidentical thinking parts.

=== query_multi() ===
Subqueries: ['What is the thinking animal argument?']
The thinking animal argument is a philosophical argument that suggests the following:

1. (P1) Presently sitting in your chair is a human animal.
2. (P2) The human animal sitting in your chair i

#### The simple-question regression is fixed, confirmed in the full notebook

Before this fix, the multiquery pipeline had a real problem on exactly this kind of question. The original version of `generate_subqueries()` worked by asking the model to write out reformulated questions in free text, one per line, and then just counting however many lines came back, there was no explicit checkpoint anywhere in that process where the model had to commit to how many genuinely distinct topics a question actually contained. That worked fine on composite questions, where there were real separate topics to split apart, but it broke down on simple ones. When this same thinking-animal question was first tested, the model split it into five separate subqueries anyway, one for its premises, one for who proposed it, one for its context, one for its criticisms, plus the original question as a fifth, even though there was really only one topic being asked about here, the argument itself. The prompt at the time already said explicitly that a single-topic question only needed one reformulation, but that instruction alone wasn't enough to stop the model from defaulting into a habitual pattern of producing several items regardless of whether the question actually called for it. The practical cost of that was concrete and measurable: the resulting answer, built from five diluted subqueries instead of one well-targeted one, dropped the philosopher names and the specific criticism that the single-query baseline included, and replaced them with a vaguer, less complete summary.

The fix that followed wasn't another rewrite of the prompt wording, since that had already been tried once and only partially worked. Instead, the decomposition step was rebuilt to force the model into a structured output with two fields, a list of the distinct topics it identifies in the question, and a list of subqueries the same length as that list, generated one per topic. This closes off the specific failure mode from before, because there's no longer a separate, implicit decision about how many lines to write sitting apart from the decision about what the topics actually are. A single-topic question can only ever produce one subquery under this design, because there's only one topic for the model to name in the first place.

Running the thinking-animal control question through the full notebook now, not just the isolated standalone script it was first verified against, `query_multi()` generates exactly one subquery, matching the original question precisely, with no splitting of the argument's premises, proposer, and criticisms into separate topics the way it did before. The resulting answer is back to full quality and, if anything, slightly sharper than the single-query baseline on this particular run: it correctly names the four premises, credits the argument to Snowdon, Carter, McDowell, Ayers, and Olson, and states the actual criticism precisely, that the view risks implying identity with an infinite number of nonidentical thinking parts, which is the same substantive criticism the baseline gave, just phrased slightly more precisely. The regression is resolved, and resolved in a way that held up outside the narrow test case it was originally fixed against, which matters more than the fix itself.

## Test 4: the African Sage Philosophy drift case

This question showed measurable accuracy drift under both naive and
rerank in the main eval (scored 3/2 on Accuracy) -- the answer conflated
Oruka's actual three-claims list (Section 1) with a related but
distinct discussion elsewhere in the article. Worth checking whether
retrieving Section 1 more directly via a targeted sub-query fixes the
conflation.


In [10]:
drift_question = "What three negative claims about African philosophy was Oruka trying to counter?"

result_drift_single = rag.query(drift_question)
result_drift_multi = rag.query_multi(drift_question, max_subqueries=5)

print("=== query() ===")
print(result_drift_single["answer"])
print("Sources:", [(s.get("title"), s.get("section")) for s in result_drift_single["sources"]])
print()
print("=== query_multi() ===")
print("Subqueries:", result_drift_multi["subqueries"])
print(result_drift_multi["answer"])
print("Sources:", [(s.get("title"), s.get("section")) for s in result_drift_multi["sources"]])


=== query() ===
Oruka aimed to counter the following three negative claims about African philosophy:

1. **Philosophical Unanimity**: Ethnophilosophy had popularized the false view that traditional Africa was a place of philosophical unanimity, suggesting that African traditions encouraged unanimity regarding beliefs and values, leaving no room for individual thinkers with independent views.

2. **Anonymity of Indigenous Thought**: There was a false perception, partly created by ethnophilosophy, that indigenous African thought is anonymous, implying that it is deeply grounded in mythical representations of reality and lacks clearly thought-out and logically valid philosophical ideas from individual thinkers.

3. **Need for a Conceptual Leap from Myth**: Oruka contested the idea that Africans needed to make a qualitative mental leap from myth to embrace philosophical thought, a view expressed by Franz Crahay, which suggested that African thought was not already separate from myths.
Sour

#### Multiquery reproduces the same drift as single-query, unchanged

This is a different kind of result from the Cicero and thinking-animal tests, worth reading carefully rather than filing as either a win or a loss for multiquery. Both `query()` and `query_multi()` give essentially the same answer here, word for word close in places, and that answer is the same substantive drift that was already documented earlier in the project's eval work on this exact question. The real three claims Oruka set out to counter, stated plainly in the article's own "Oruka's Project" section, are that Africans supposedly don't reason the way the Greeks did, that oral traditions can't produce real philosophy without literacy, and that African traditions enforce a unanimity that discourages individual critical thought. What both methods actually retrieved and answered from here is a different, adjacent list from a different section of the same article, the ethnophilosophy discussion's own three false claims about unanimity, anonymity, and the supposed need for a mental leap from myth to philosophy. That's a real, coherent set of claims that genuinely appears in the source text, so neither method is hallucinating anything, but it's not the specific three claims the question actually asked about, and both methods land on the identical wrong-but-adjacent answer independently.

What makes this worth logging carefully is what it says about the limits of what multiquery actually fixes. Look at the subqueries generated here: two of them, and both are close paraphrases of the original question rather than a real decomposition into separate topics. That's not a bug in the topic-first structural fix, it's the fix working exactly as intended, since this question genuinely only has one topic, "what are Oruka's three claims," not several distinct things that would benefit from being searched separately. The problem here was never that a single embedding had to represent two competing topics at once, which is the specific failure mode multiquery was built to solve. The problem is that the correct source section, "Oruka's Project," simply isn't ranking highly enough against a topically similar but substantively different section, regardless of how the question gets phrased or how many times it gets searched. Multiquery doesn't touch that kind of failure at all, because there's no second topic to split off and target separately, there's just one topic whose best-matching chunk in the index happens to be the wrong one.

## Test 5: Anselm -- mercy/justice vs. freedom/sin

Cross-section composite within a single confirmed-indexed article
(verified directly against the exported article text). Section 3
covers how Anselm reconciles God's mercy and justice; section 4 covers
his account of freedom, sin, and redemption -- distant sections, same
article, no cross-article risk.


In [11]:
q5 = "How does Anselm reconcile God's mercy and justice, and how does that account connect to his view of human freedom and the possibility of sin?"

r5_single = rag.query(q5)
r5_multi = rag.query_multi(q5, max_subqueries=5)

print("=== query() ===")
print(r5_single["answer"])
print()
print("=== query_multi() ===")
print("Subqueries:", r5_multi["subqueries"])
print(r5_multi["answer"])


=== query() ===
Anselm reconciles God's mercy and justice by appealing to God's supreme goodness. He argues that it is better for God to be good to both the good and the wicked, rather than only to the good. This goodness is expressed in both punishing and sparing the wicked, rather than only punishing them. Anselm further argues that justice itself requires mercy because God's justice to himself requires exercising his supreme goodness in sparing the wicked. Thus, in saving those who might justly be destroyed, God is just not because he gives them their due, but because he acts in accordance with his supreme goodness.

This account connects to Anselm's view of human freedom and the possibility of sin through the concept of self-initiated action. In Anselm's view, the angels were given both a will for happiness and a will for justice, allowing them the power for self-initiated action. This freedom meant that their choices, whether to pursue justice or happiness, originated from themsel

#### Both methods succeed, multiquery slightly more precise

This composite question spans two sections of the same article, God's mercy and justice in one, and freedom and the possibility of sin in another, and both `query()` and `query_multi()` handled it well, producing correct, complete, and well-reasoned answers that genuinely connect the two halves rather than treating them separately. Both correctly identify that God's supreme goodness requires exercising mercy alongside justice, and both tie that back to the angels having been given both a will for happiness and a will for justice, which is what gave them real self-initiated choice in the first place.

The subqueries generated here show the topic-first fix doing exactly what it's supposed to on a question that actually calls for it, four sub-questions covering the reconciliation of mercy and justice, Anselm's view of freedom, and his view of sin, each a genuinely distinct sub-topic rather than a repeated paraphrase of the whole question the way test 4's did. The two final answers differ mostly in framing rather than substance. The single-query answer centers on the idea of self-initiated action as the connecting thread, while the multiquery answer leads with rectitude of will, which is Anselm's own term for what freedom actually preserves, before arriving at self-initiated action as well. Neither framing is wrong, but the multiquery version sits slightly closer to how Anselm's own definition of freedom is actually built, since rectitude of will is the more foundational concept in his account.

Worth noting plainly that this is a different kind of result than the Cicero test: here, single-query retrieval already handled the question well on its own, so multiquery's contribution was a modest refinement rather than a fix for an outright failure. Combined with test 4's negative result, a real pattern is starting to take shape, multiquery's advantage isn't a blanket improvement across every composite question, it shows up specifically where single-query retrieval already struggles for that particular question, and does little or nothing where it doesn't.

## Test 6: Abelard -- universals vs. intentions

Cross-section composite within Peter Abelard (section 2, metaphysics of
universals; section 6, ethics of intentions) -- verified against the
same source text used in Part 1 of the blog.


In [12]:
q6 = "How does Abelard's rejection of universals as real things relate to his view that intentions, not deeds, determine moral worth?"

r6_single = rag.query(q6)
r6_multi = rag.query_multi(q6, max_subqueries=5)

print("=== query() ===")
print(r6_single["answer"])
print()
print("=== query_multi() ===")
print("Subqueries:", r6_multi["subqueries"])
print(r6_multi["answer"])


=== query() ===
The context does not provide information about Abelard's rejection of universals as real things, so it is not possible to relate this to his view that intentions, not deeds, determine moral worth. The context focuses solely on Abelard's intentionalist ethics, where the moral worth of an action is determined by the agent's intention rather than the consequences or the deeds themselves. Without information on his stance on universals, a connection cannot be made.

=== query_multi() ===
Subqueries: ["What is Abelard's rejection of universals as real things?", 'How does Abelard believe intentions determine moral worth?', "How does Abelard's rejection of universals as real things relate to his view that intentions, not deeds, determine moral worth?"]
Abelard's rejection of universals as real things is consistent with his view that intentions, not deeds, determine moral worth, in that both positions emphasize the importance of individual, particular instances over abstract, g

#### A clean multiquery win, and a genuinely interesting synthesis

Single-query retrieval failed outright on this one, and it did so honestly rather than by hallucinating around the gap. The answer states plainly that the context only contains Abelard's intentionalist ethics and nothing about his rejection of universals, so no connection can be drawn, which is exactly the correct response to a genuine retrieval gap, one embedding representing two topics, and one of them apparently losing out entirely.

`query_multi()` retrieved both halves cleanly, and what it did with them is worth pausing on, since it's more than just stating fact A next to fact B. It identified a real structural parallel between Abelard's metaphysics and his ethics that isn't spelled out explicitly in either retrieved chunk on its own, that both positions share a preference for the particular over the abstract. Universals, on Abelard's view, are merely words rather than real entities standing over concrete individuals, and moral worth similarly lives in the individual agent's particular intention rather than in some general category of what the deed itself was. That's a genuine piece of philosophical synthesis, connecting two separately-stated ideas into an insight neither one states on its own, which is exactly the kind of reasoning the earlier generation-prompt fix was meant to unlock, not just permitting the model to state two facts side by side, but to actually think about how they relate.

This fits the pattern building across the last few tests: multiquery's real advantage keeps showing up specifically in cases where single-query retrieval concretely fails to represent both halves of a question, the same shape of failure as the Cicero case, and this one is arguably the cleanest example of it yet.

## Test 7: Animalism -- thinking animal argument vs. organic/somatic distinction

Cross-section composite within Animalism (section 3.1, the thinking
animal argument; section 1.2, organic vs. somatic animalism) -- this is
the same article that produced a confirmed rerank failure in the main
eval, worth checking whether multiquery handles it differently.


In [13]:
q7 = "How does the thinking animal argument support animalism, and how does that connect to the distinction between organic and somatic versions of the view?"

r7_single = rag.query(q7)
r7_multi = rag.query_multi(q7, max_subqueries=5)

print("=== query() ===")
print(r7_single["answer"])
print()
print("=== query_multi() ===")
print("Subqueries:", r7_multi["subqueries"])
print(r7_multi["answer"])


=== query() ===
The context does not provide information about the distinction between organic and somatic versions of animalism, nor does it explicitly connect the thinking animal argument to these versions. The thinking animal argument is mentioned as a point of contention, suggesting that it should be rejected because it leads to the problematic conclusion that a person could be identical with an infinite number of nonidentical thinking parts. However, the context does not elaborate on how this argument supports animalism or how it relates to any specific versions of the view.

=== query_multi() ===
Subqueries: ['What is the thinking animal argument?', 'How does the thinking animal argument support animalism?', 'What is the distinction between organic and somatic versions of animalism?', 'How does the thinking animal argument support animalism, and how does that connect to the distinction between organic and somatic versions of the view?']
The thinking animal argument supports anima

#### Another clean multiquery win, though a thinner synthesis than test 6

Single-query retrieval failed here in the same honest way it did on the Abelard question, it found the thinking animal argument content but not the organic and somatic distinction, and correctly said it couldn't connect them rather than inventing a link. Worth noting this is the same article that produced a confirmed rerank failure earlier in the project's main eval on this exact organic-versus-somatic question, which suggests this particular article is a recurring source of retrieval trouble across different pipeline configurations, not just a one-time fluke tied to one specific method.

`query_multi()` retrieved both halves and did attempt to connect them, but the connection it draws is real without being especially deep. It correctly says both the thinking animal argument and the organic and somatic debate concern the conditions of animal continuity and the nature of human animals, which is true and relevant, but it doesn't actually answer the more substantive version of the question, whether the thinking animal argument itself favors one side of the organic and somatic split over the other, or whether it's neutral between them. Nothing here is fabricated, and the answer is accurate as far as it goes, but it reads more like a description of how the two topics are thematically

## Tests 8-9: extreme demo stress tests -- NOT part of the validated eval

Everything above is a genuine test question, built the same way as the
scored naive-vs-rerank eval: written in advance, verified against real
indexed content. These last two are different in kind, and it matters
to keep that distinction visible: they're deliberately extreme,
many-part synthesis questions built to demonstrate a contrast against
ungrounded general-purpose chatbots (ChatGPT, Claude, Gemini used
directly, with no access to this corpus) and against single-query RAG.
They are NOT scored against a pre-written expected-answer key the way
the real eval questions are, so treat their output as illustrative, not
as evidence with the same weight as the tests above. If these get used
in a demo or a follow-up post, that distinction should stay explicit
there too.

Every claim referenced in both questions is drawn from verified,
indexed content across all 10 of the confirmed articles, the same
standard as the real tests, the only thing that's different here is
scale and the lack of a pre-written scoring rubric. Both questions use
a much higher `max_subqueries` cap than the default -- this is exactly
the kind of case the dynamic-count design in `multiquery.py` was built
for, letting the model decide it genuinely needs many reformulations
rather than being capped at a small fixed number.

Worth running the same two questions through a plain ChatGPT/Claude/
Gemini session yourself for the actual side-by-side comparison -- that
contrast, not this notebook's output alone, is what makes the case for
multiquery-plus-grounding land.


In [15]:
q8 = (
    "I've got a handful of things I'm curious about across the Stanford "
    "Encyclopedia of Philosophy and I'd love one unified answer that "
    "pulls them all together. First, why did Abelard think universals "
    "weren't real things, and how does that connect to his idea that a "
    "person's intentions, rather than their actual deeds, are what "
    "determine moral worth? I've also always been a bit fuzzy on the real "
    "difference between induction and abduction as forms of inference -- "
    "could you clear that up? On the aesthetics side, what's the "
    "disinterest thesis, and how did Kant end up reinterpreting it? I'm "
    "also interested in the Bakke affirmative action case specifically -- "
    "which reasons did Justice Powell accept, and which did he reject? "
    "Switching over to African philosophy, what were the three negative "
    "claims Oruka was trying to push back against? On Islamic philosophy, "
    "why did al-Farabi think metaphysics wasn't really a theological "
    "science? For Anselm, how does he manage to reconcile God's mercy "
    "with God's justice? And on animalism, what's the actual thinking "
    "animal argument? Finally, why did Cicero think Rome under the "
    "Republic counted as a legitimate res publica?"
)

r8_single = rag.query(q8)
r8_multi_rerank = rag.query_multi(q8, max_subqueries=12, use_rerank=True)
r8_multi_naive = rag.query_multi(q8, max_subqueries=12, use_rerank=False)

print(f"Number of subqueries generated: {len(r8_multi_rerank['subqueries'])}")
print("Subqueries:", r8_multi_rerank["subqueries"])
print()
print("=== query() (single embedding) ===")
print(r8_single["answer"])
print()
print("=== query_multi(), reranked ===")
print(r8_multi_rerank["answer"])
print()
print("=== query_multi(), no rerank ===")
print(r8_multi_naive["answer"])

Number of subqueries generated: 10
Subqueries: ["Why did Abelard think universals weren't real things, and how does that connect to his idea that a person's intentions determine moral worth?", 'What is the difference between induction and abduction as forms of inference?', 'What is the disinterest thesis, and how did Kant reinterpret it?', 'What reasons did Justice Powell accept and reject in the Bakke affirmative action case?', 'What were the three negative claims Oruka was trying to push back against?', "Why did al-Farabi think metaphysics wasn't really a theological science?", "How does Anselm reconcile God's mercy with God's justice?", 'What is the thinking animal argument?', 'Why did Cicero think Rome under the Republic counted as a legitimate res publica?', "I've got a handful of things I'm curious about across the Stanford Encyclopedia of Philosophy and I'd love one unified answer that pulls them all together. First, why did Abelard think universals weren't real things, and how 

#### Test 8 (10 topics, natural prose): multiquery is what makes this question answerable at all

The headline result here is multiquery succeeding, not a rerank comparison. Single-query retrieval, working from one embedding trying to represent ten genuinely distinct topics at once, could only ever land on one of them, Abelard, and openly admitted it couldn't address the rest. Multiquery decomposition is what made the other nine topics retrievable in the first place, correctly identifying all ten distinct topics from a genuinely flowing, unmarked paragraph, with no explicit numbering to lean on. That's the real finding: without decomposition, nine-tenths of this question simply had no path to an answer.

Once decomposition correctly found all ten topics, the two ways of using that decomposition, reranked versus simply concatenated, both produced real, mostly accurate coverage of most of the ten. The gap between those two was secondary to the much larger gap between having decomposition at all and not having it.

In [16]:
q9 = (
    "I've got a longer list of things I'm curious about, all from the "
    "Stanford Encyclopedia of Philosophy, and I'd like one unified answer "
    "covering everything. Starting with Abelard: why did he think "
    "universals weren't real things, and how does that connect to his "
    "view that a person's intentions, not their deeds, determine moral "
    "worth? On the logic side, what's the actual difference between "
    "induction and abduction, and separately, what's the argument of the "
    "bad lot, and how does it challenge the idea that the best "
    "explanation among the candidates you've considered is really the "
    "best explanation available? Moving to aesthetics, what's the "
    "disinterest thesis and how did Kant end up reinterpreting it, and "
    "why does Danto think Warhol's Brillo Boxes undermines artistic "
    "formalism? On the legal side, which reasons did Justice Powell "
    "accept and which did he reject in the Bakke affirmative action case, "
    "and how did the 2023 SFFA v. Harvard decision end up differing from "
    "Bakke? Turning to African philosophy, what were the three negative "
    "claims Oruka was trying to counter, and what actually distinguishes "
    "a folk sage from a philosophic sage? On alienation, what's the "
    "difference between the subjective and objective versions of it, and "
    "separately, how does alienation differ from fetishism? Switching to "
    "Islamic philosophy, why did al-Farabi think metaphysics wasn't "
    "really a theological science? For Anselm, how does he reconcile "
    "God's mercy with God's justice, and what was Gaunilo's Lost Island "
    "objection to the ontological argument? On animalism, what's the "
    "actual thinking animal argument, and what's the difference between "
    "the organic and somatic versions of the view? And finally, why did "
    "Cicero think Rome under the Republic counted as a legitimate res "
    "publica?"
)

r9_single = rag.query(q9)
r9_multi_rerank = rag.query_multi(q9, max_subqueries=20, use_rerank=True)
r9_multi_naive = rag.query_multi(q9, max_subqueries=20, use_rerank=False)

print(f"Number of subqueries generated: {len(r9_multi_rerank['subqueries'])}")
print("Subqueries:", r9_multi_rerank["subqueries"])
print()
print("=== query() (single embedding) ===")
print(r9_single["answer"])
print()
print("=== query_multi(), reranked ===")
print(r9_multi_rerank["answer"])
print()
print("=== query_multi(), no rerank ===")
print(r9_multi_naive["answer"])

Number of subqueries generated: 18
Subqueries: ["What was Abelard's view on universals and how does it relate to his moral philosophy regarding intentions?", 'What is the difference between induction and abduction in logic?', 'What is the argument of the bad lot and how does it challenge the notion of the best explanation?', 'What is the disinterest thesis and how did Kant reinterpret it?', "Why does Danto believe Warhol's Brillo Boxes undermine artistic formalism?", 'What reasons did Justice Powell accept and reject in the Bakke affirmative action case?', 'How did the 2023 SFFA v. Harvard decision differ from the Bakke case?', 'What were the three negative claims Oruka aimed to counter in African philosophy?', 'What distinguishes a folk sage from a philosophic sage in African philosophy?', 'What is the difference between subjective and objective alienation?', 'How does alienation differ from fetishism?', 'Why did al-Farabi believe metaphysics is not a theological science?', "How does 

#### Test 9 (18 topics, natural prose): multiquery succeeds at real scale, one implementation of it doesn't

Decomposition correctly identified all eighteen distinct topics from natural, unmarked prose, the clearest confirmation yet that the structural fix genuinely scales, not just to two or three topics but to eighteen. That's the core multiquery claim holding up under real pressure.

What varied was what happened *after* decomposition succeeded. Single-query retrieval, with no decomposition to work from, answered one topic out of eighteen and, worse than in the ten-topic case, didn't even flag the other seventeen as missing. Reranking the pooled subqueries, despite having eighteen correctly-targeted retrieval passes behind it, failed completely and answered nothing, a real regression worth treating as an open problem, since the decomposition step that fed it worked perfectly well. Skipping reranking and using each subquery's own retrieved slice directly delivered real, mostly accurate coverage of all eighteen topics. The success belongs to multiquery's decomposition, which worked in every version of this test; the failure belongs specifically to how the reranked variant used the results decomposition correctly produced.